####Setup

In [8]:
import pandas as pd
import numpy as np
import pandas as pd
from numpy.typing import ArrayLike
import kagglehub
from typing import Tuple, List

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor # replace GradientBoostingRegressor with this
from sklearn.svm import SVR, LinearSVR

from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import requests
from io import StringIO
from datetime import datetime, timedelta

####Data Collection and Preprocessing

In [9]:
def get_weather_data(city: str) -> ArrayLike:
    """
    Downloads weather data for a given city from the Kaggle dataset and scales
    the daily average temperatures using StandardScaler.

    Parameters:
    -----------
    city : str
        The city for which to download weather data.

    Returns:
    --------
    np.ndarray
        1D NumPy array of scaled daily average temperatures.
    """
    path = kagglehub.dataset_download("gucci1337/weather-of-albania-last-three-years")
    years = [2021, 2022, 2023]
    data_frames = []

    for year in years:
        file_path = f"{path}/data_weather/{city}/{city}{year}.csv"
        df = pd.read_csv(file_path)
        df = df.dropna(subset=['tavg'])  # Remove rows where 'tavg' is NaN
        data_frames.append(df['tavg'])

    # Concatenate the 'tavg' columns from each year's DataFrame
    concatenated_data = pd.concat(data_frames, ignore_index=True)

    # Scale the data using StandrdScaler
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(concatenated_data.values.reshape(-1, 1)).flatten()

    return scaled_data

def create_forecasting_dataset(data: np.ndarray, n_input=256, n_output=128):
    """
    Transforms a 1D array `data` into supervised learning samples (X, y)
    where each X is `n_input` days long and each y is the subsequent `n_output` days.

    Parameters:
    -----------
    data : np.ndarray
        1D NumPy array of scaled time-series data (e.g., daily temperatures).
    n_input : int, optional
        Number of past days used as input.
    n_output : int, optional
        Number of future days to forecast.

    Returns:
    --------
    X : np.ndarray
        2D array of shape (num_samples, n_input).
    y : np.ndarray
        2D array of shape (num_samples, n_output).
    """
    X, y = [], []
    max_start = len(data) - n_input - n_output + 1
    for i in range(max_start):
        X.append(data[i : i + n_input])
        y.append(data[i + n_input : i + n_input + n_output])
    return np.array(X), np.array(y)

In [10]:
def get_energy_data(year: int):
    """
    Downloads daily consumption data for a given year, concatenates all days into a single DataFrame,
    and performs basic preprocessing (e.g., dropping NaN values). The returned data is scaled
    using MinMaxScaler.
    """
    all_data = []
    current_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    while current_date <= end_date:
        date_str = f"{current_date.day}.{current_date.month}.{current_date.year}"
        url = f"https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t{date_str}"
        print(f"Fetching data for {date_str} from:\n{url}")
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an error for bad status codes
            daily_df = pd.read_csv(StringIO(response.text), sep=';')
            all_data.append(daily_df)
        except Exception as e:
            print(f"Error fetching data for {date_str}: {e}")
        current_date += timedelta(days=1)
    combined_data = pd.concat(all_data, ignore_index=True)
    # Assume the consumption values are in the second column
    string_values = combined_data.iloc[:, 1].values
    values_float = np.array([float(w.replace(',', '')) for w in string_values])

    # Scale the data using MinMaxScaler
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(values_float.reshape(-1, 1)).flatten()

    return scaled_values


In [11]:
def get_new_cases_by_country(df: pd.DataFrame) -> dict:
    """
    Groups the DataFrame by 'Country', sorts by 'Date_reported', and returns a dictionary
    mapping country names to a NumPy array of new cases.
    """
    result = {}
    for country, group in df.groupby('Country'):
        group_sorted = group.sort_values('Date_reported')
        new_cases_array = group_sorted['New_cases'].to_numpy()
        result[country] = new_cases_array
    return result

def get_healthcare_data(country: str) -> np.ndarray:
    """
    Returns a NumPy array of new COVID-19 cases for the specified country.
    Data is taken from the WHO global daily dataset, and a slice [200:1200] is returned.
    The returned data is scaled to the [0, 1] range using MinMaxScaler.
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/WHO-COVID-19-global-daily-data.csv"
    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)
    df = df.sort_values(['Country', 'Date_reported'])
    # Interpolate missing values in the 'New_cases' column for each country
    df['New_cases'] = df.groupby('Country')['New_cases'].transform(lambda group: group.interpolate(method='linear'))
    cases_dict = get_new_cases_by_country(df)
    new_cases = cases_dict.get(country)
    if new_cases is None:
        raise ValueError(f"Country '{country}' not found in dataset")

    # Slice the data and apply MinMax scaling
    slice_data = new_cases[200:1200]
    scaler = StandardScaler()
    scaled_slice = scaler.fit_transform(slice_data.reshape(-1, 1)).flatten()
    return scaled_slice

In [12]:
def get_finance_data():
    """
    Returns a numpy array containing the finance data, scaled to the [0, 1] range.
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/AMZN-stock-price.csv"
    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)
    # Assuming the second column holds the desired data.
    df = df.iloc[:, 1]
    values = df.values
    # Scale the data using MinMaxScaler
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(values.reshape(-1, 1)).flatten()
    return scaled_values

#### Training and Testing

In [13]:
from time import time
import warnings
warnings.filterwarnings("ignore")  # suppress any convergence warnings for brevity

def mean_absolute_scaled_error(y_true, y_pred, training_series, seasonality=1):
    """
    Computes the Mean Absolute Scaled Error (MASE).

    Parameters:
        y_true (np.ndarray): True target values.
        y_pred (np.ndarray): Predicted target values.
        training_series (np.ndarray): The original time-series data used for training.
        seasonality (int): Seasonal period (default 1 for a naïve one-step forecast).

    Returns:
        float: The MASE value.
    """
    # Compute the scaling factor using the naïve forecast errors on the training series.
    # Avoid division by zero by substituting a very small number if necessary.
    d = np.mean(np.abs(np.diff(training_series, n=seasonality)))
    if d == 0:
        d = np.finfo(np.float64).eps

    errors = np.abs(y_true - y_pred)
    return np.mean(errors) / d

def multi_model_forecaster(data: np.ndarray, train_ratio=0.6, n_input=256, n_output=128):
    """
    1. Converts a 1D array of time-series data into (X, y) blocks with the specified
       input and output lengths.
    2. Splits chronologically into training and testing sets.
    3. Trains multiple multi-output regressors on the training set.
    4. Evaluates each on the test set and returns a dict of (model, metrics).

    Parameters
    ----------
    data : np.ndarray
        1D array of time-series data.
    train_ratio : float
        Fraction of samples to use for training (chronologically).
    n_input : int
        Number of past days used as input features.
    n_output : int
        Number of future days to predict.

    Returns
    -------
    results : dict
        A dictionary where each key is the model name and each value is another dict:
        {
            "model": the fitted model,
            "mae": mean absolute error on test,
            "rmse": root mean squared error on test,
            "mase": mean absolute scaled error on test,
            "X_train": X_train,
            "y_train": y_train,
            "X_test": X_test,
            "y_test": y_test,
            "y_pred": predictions on test
        }
    """
    # ----------------------
    # 1. Create the dataset
    # ----------------------
    X, y = create_forecasting_dataset(data, n_input=n_input, n_output=n_output)

    # ---------------------------
    # 2. Chronological train-test split
    # ---------------------------
    num_samples = X.shape[0]
    train_size = int(num_samples * train_ratio)

    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    # For MASE, we compute the scaling factor on the in-sample (training) portion of the original series.
    # We use the portion of data used to create the training inputs.
    training_series_for_mase = data[:train_size + n_input]

    # ---------------------------------------------------------
    # 3. Define the models you want to train and evaluate
    # ---------------------------------------------------------
    models = {
        "LinearRegression": LinearRegression(),
        "RandomForestRegressor": RandomForestRegressor(
            n_estimators=10, random_state=42, n_jobs=-1
        ),
        "ExtraTreesRegressor": ExtraTreesRegressor(
            n_estimators=10, random_state=42, n_jobs=-1
        ),
        "MLPRegressor": MLPRegressor(
            hidden_layer_sizes=(100, 100),
            max_iter=10,
            random_state=42
        ),
        "Linear SVR": MultiOutputRegressor(LinearSVR(max_iter=10)),
        "HistGradientBoostingRegressor": MultiOutputRegressor(
            HistGradientBoostingRegressor(max_iter=10, random_state=42)
        )
    }

    results = {}

    for model_name, model in models.items():
        # 3a. Fit the model on the training data
        start = time()
        print("Training:", model_name, "...", end=" ")
        model.fit(X_train, y_train)
        elapsed = time() - start
        print(f"Done in {elapsed:.2f} seconds.")

        # 3b. Predict on the test set
        y_pred = model.predict(X_test)

        # 3c. Evaluate with MAE, RMSE, and MASE
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        mase = mean_absolute_scaled_error(y_test, y_pred, training_series=training_series_for_mase, seasonality=1)

        # Store everything in a results dict
        results[model_name] = {
            "model": model,
            "mae": mae,
            "rmse": rmse,
            "mase": mase,
            "X_train": X_train,
            "y_train": y_train,
            "X_test": X_test,
            "y_test": y_test,
            "y_pred": y_pred
        }

    return results

In [14]:
from time import time
import warnings
warnings.filterwarnings("ignore")  # suppress any convergence warnings for brevity

# Define the horizon pairs: (n_input, n_output)
horizon_pairs = [(64, 32), (128, 64), (256, 128)]

# Download the datasets.
city_name = "lezhe"
weather_data = get_weather_data(city_name)

year = 2024
energy_data = get_energy_data(year)

finance_data = get_finance_data()

# For healthcare data, provide a country name (e.g., "USA"); adjust as needed.
healthcare_data = get_healthcare_data("Germany")

# Collect datasets into a dictionary.
datasets = {
    "weather": weather_data,
    "energy": energy_data,
    "finance": finance_data,
    "healthcare": healthcare_data,
}

# Container to store all results.
all_results = {}

# Iterate over each dataset and each horizon pair.
for ds_name, data in datasets.items():
    print(f"\n===== Dataset: {ds_name} =====")
    all_results[ds_name] = {}
    for n_input, n_output in horizon_pairs:
        print(f"\n--- Horizon Pair: Past = {n_input}, Future = {n_output} ---")
        start_time = time()
        results = multi_model_forecaster(data=data, train_ratio=0.6, n_input=n_input, n_output=n_output)
        elapsed = time() - start_time
        print(f"Forecasting completed in {elapsed:.2f} seconds.\n")
        # Print out summary metrics for each model.
        for model_name, metrics in results.items():
            print(f"Model: {model_name}")
            print(f"  MAE  : {metrics['mae']:.4f}")
            print(f"  RMSE : {metrics['rmse']:.4f}")
            print(f"  MASE : {metrics['mase']:.4f}\n")
        # Save results for this horizon pair.
        all_results[ds_name][(n_input, n_output)] = results


Fetching data for 1.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t1.1.2024
Fetching data for 2.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t2.1.2024
Fetching data for 3.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t3.1.2024
Fetching data for 4.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t4.1.2024
Fetching data for 5.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t5.1.2024
Fetching data for 6.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t6.1.2024
Fetching data for 7.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t7.1.2024
Fetching data for 8.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t8.1.2024
Fetching data for 9.1.2024 from:
https://www.eview.de/e1/p3Export.php?fr

####Evaluation

In [15]:
def print_model_metrics(results):
    """
    Iterate over the results dictionary and display the model's name,
    along with the RMSE and MAE.
    """
    for model_name, info in results.items():
        rmse = info["rmse"]
        mae = info["mae"]
        mase = info["mase"]
        print(f"Model: {model_name}\tRMSE: {rmse:.4f}\tMAE: {mae:.4f}\tMASE: {mase:.4f}")

print_model_metrics(results)

Model: LinearRegression	RMSE: 18.8402	MAE: 16.4885	MASE: 991.2257
Model: RandomForestRegressor	RMSE: 1.8402	MAE: 1.6899	MASE: 101.5923
Model: ExtraTreesRegressor	RMSE: 1.2634	MAE: 1.1521	MASE: 69.2597
Model: MLPRegressor	RMSE: 1.0400	MAE: 0.8744	MASE: 52.5646
Model: Linear SVR	RMSE: 10.6559	MAE: 9.1243	MASE: 548.5183
Model: HistGradientBoostingRegressor	RMSE: 0.8774	MAE: 0.7875	MASE: 47.3437


####Plotting

In [18]:
import os
import matplotlib.pyplot as plt

def save_forecast_plot_single_model(results, model_name, dataset_name, horizon, sample_index=0):
    """
    Saves the forecast plot for a single model to a file in the directory plots/model/.
    The file is named as dataset_horizon.png (e.g., weather_64-32.png for a 64-day input and 32-day output).
    """
    info = results[model_name]
    X_test = info["X_test"]
    y_test = info["y_test"]
    y_pred = info["y_pred"]

    # Extract the specific sample
    x_hist = X_test[sample_index]   # shape: (n_input,)
    y_true = y_test[sample_index]   # shape: (n_output,)
    y_hat = y_pred[sample_index]    # shape: (n_output,)

    n_input = len(x_hist)
    n_output = len(y_true)

    # Build x-axes
    hist_x_axis = range(n_input)
    future_x_axis = range(n_input, n_input + n_output)

    plt.figure(figsize=(8, 5))
    plt.plot(hist_x_axis, x_hist, label="Historical", color="blue")
    plt.plot(future_x_axis, y_true, label="Actual Future", color="green")
    plt.plot(future_x_axis, y_hat, label=f"Predicted ({model_name})", color="red")

    plt.title(f"Forecast for Sample #{sample_index} - Model: {model_name}")
    plt.xlabel("Days")
    plt.ylabel("Value")
    plt.legend()
    plt.tight_layout()

    # Create directory for the model if it doesn't exist
    save_dir = os.path.join("plots", model_name)
    os.makedirs(save_dir, exist_ok=True)

    # Format the horizon as "n_input-n_output" for the file name
    horizon_str = f"{horizon[0]}-{horizon[1]}"
    file_name = f"{dataset_name}_{horizon_str}.png"
    file_path = os.path.join(save_dir, file_name)

    plt.savefig(file_path)
    plt.close()
    print(f"Saved forecast plot for {model_name} (dataset: {dataset_name}, horizon: {horizon_str}) to {file_path}")

# Call the method for each model, dataset, and horizon pair.
# Assuming 'all_results' is the container holding your forecast results.
for dataset_name, horizons_dict in all_results.items():
    for horizon, results in horizons_dict.items():
        for model_name in results.keys():
            save_forecast_plot_single_model(results, model_name, dataset_name, horizon, sample_index=0)


Saved forecast plot for LinearRegression (dataset: weather, horizon: 64-32) to plots/LinearRegression/weather_64-32.png
Saved forecast plot for RandomForestRegressor (dataset: weather, horizon: 64-32) to plots/RandomForestRegressor/weather_64-32.png
Saved forecast plot for ExtraTreesRegressor (dataset: weather, horizon: 64-32) to plots/ExtraTreesRegressor/weather_64-32.png
Saved forecast plot for MLPRegressor (dataset: weather, horizon: 64-32) to plots/MLPRegressor/weather_64-32.png
Saved forecast plot for Linear SVR (dataset: weather, horizon: 64-32) to plots/Linear SVR/weather_64-32.png
Saved forecast plot for HistGradientBoostingRegressor (dataset: weather, horizon: 64-32) to plots/HistGradientBoostingRegressor/weather_64-32.png
Saved forecast plot for LinearRegression (dataset: weather, horizon: 128-64) to plots/LinearRegression/weather_128-64.png
Saved forecast plot for RandomForestRegressor (dataset: weather, horizon: 128-64) to plots/RandomForestRegressor/weather_128-64.png
Save